# Secure Processing Environment

This notebook runs inside an isolated environment. It can **only** access the data covered by your permit.

**No internet access is available** — any external requests will fail by design.

In [ ]:
import os
import requests
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(os.environ['DATABASE_URL'])
PERMIT_ID = os.environ.get('PERMIT_ID', 'unknown')
LLM_GATEWAY_URL = os.environ.get('LLM_GATEWAY_URL', 'http://host.docker.internal:8006')
print('Connected. Permit:', PERMIT_ID)

In [ ]:
# List available views in your permit schema
with engine.connect() as conn:
    rows = conn.execute(text("""
        SELECT table_name FROM information_schema.views
        WHERE table_schema = (SELECT setting FROM pg_settings WHERE name = 'search_path')
    """)).fetchall()
available_views = [r[0] for r in rows]
print('Available views:', available_views)

In [ ]:
def ask_assistant(question: str) -> str:
    """Ask the LLM copilot to generate analysis code scoped to your permit's views."""
    resp = requests.post(
        f"{LLM_GATEWAY_URL}/chat/spe",
        json={
            "question": question,
            "permit_id": PERMIT_ID,
            "available_views": available_views,
            "user_id": PERMIT_ID,
        },
        timeout=60,
    )
    if not resp.ok:
        return f"Error: {resp.json().get('detail', resp.text)}"
    return resp.json()["reply"]

# Example:
# print(ask_assistant("Show me condition counts by year as a bar chart"))

In [ ]:
# Example: load conditions
with engine.connect() as conn:
    df = pd.read_sql('SELECT * FROM conditions LIMIT 5', conn)
df

In [ ]:
# Verify no internet access (this should fail)
import urllib.request
try:
    urllib.request.urlopen('https://google.com', timeout=3)
    print('WARNING: internet access available!')
except Exception as e:
    print('Good — no internet access:', e)